In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


A. EXTRACT (bobot 10%)

Baca ketiga sumber data dari HDFS (bukan disk lokal) menjadi tiga Spark DataFrame.

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("Tugas7_SparkSQL").getOrCreate()

df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")

df_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)

B. TRANSFORM — Join & Kolom Turunan (bobot 20%)

Gabungkan ketiga tabel, tambahkan kolom total_penjualan (qty x harga).

In [5]:
df_gabungan = df_transaksi \
    .join(df_cabang, on="cabang_id", how="inner") \
    .join(df_menu, on="menu_id", how="inner")

df_final = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

df_final.show(10)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


C. TRANSFORM — Window Function (bobot 20%)

Tentukan top-2 menu terlaris (berdasarkan total_penjualan) di setiap cabang, gunakan row_number().

In [6]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, row_number

df_agregasi = df_final.groupBy("nama_cabang", "nama_menu") \
    .agg(sum("total_penjualan").alias("total_penjualan_menu"))

window_spec = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan_menu").desc())

df_ranked = df_agregasi.withColumn("rank", row_number().over(window_spec))

df_top2_menu = df_ranked.filter(col("rank") <= 2)

df_top2_menu.show(truncate=False)

+-------------------------+---------+--------------------+----+
|nama_cabang              |nama_menu|total_penjualan_menu|rank|
+-------------------------+---------+--------------------+----+
|Kopi Nusantara Magelang  |Menu-1   |8505000             |1   |
|Kopi Nusantara Magelang  |Menu-12  |8085000             |2   |
|Kopi Nusantara Semarang  |Menu-13  |8750000             |1   |
|Kopi Nusantara Semarang  |Menu-9   |8715000             |2   |
|Kopi Nusantara Yogyakarta|Menu-12  |9275000             |1   |
|Kopi Nusantara Yogyakarta|Menu-15  |8160000             |2   |
+-------------------------+---------+--------------------+----+



D. TRANSFORM — Spark SQL (bobot 20%)

Menggunakan Spark SQL murni, tampilkan total penjualan per kepala_barista, diurutkan dari tertinggi.

In [7]:
from pyspark.sql.functions import col, sum

df_barista_sales = df_final.groupBy("kepala_barista") \
    .agg(sum("total_penjualan").alias("total_penjualan_barista")) \
    .orderBy(col("total_penjualan_barista").desc())

df_barista_sales.show()

+--------------+-----------------------+
|kepala_barista|total_penjualan_barista|
+--------------+-----------------------+
|         Nadia|              177680000|
|          Reza|               84645000|
+--------------+-----------------------+



E. LOAD (bobot 15%)

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path /user/[username]/tugas7/processed/laporan, dipartisi berdasarkan kategori_menu. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

In [8]:
!hdfs dfs -mkdir -p /user/amba/t7/processed/laporan
output_path = "hdfs://localhost:9000/user/amba/t7/processed/laporan"

df_final.write \
    .partitionBy("kategori_menu") \
    .mode("overwrite") \
    .parquet(output_path)

df_verify = spark.read.parquet(output_path)

df_verify.printSchema() 

df_verify.show(10)

print(f"Total baris data yang disimpan: {df_verify.count()}")

root
 |-- menu_id: integer (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- trx_id: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: integer (nullable = true)
 |-- total_penjualan: integer (nullable = true)
 |-- kategori_menu: string (nullable = true)

+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu|harga|total_penjualan| kategori_menu|
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|35000|         140000|Makanan Ringan|
|      9|        3|   KN8|  4|         Na

F. Ringkasan Eksekutif & Rekomendasi (bobot 15%)

Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (minimal 120 kata): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

In [13]:
from pyspark.sql.functions import sum, count, avg, round
from pyspark.sql.functions import col, sum

df_ringkasan_cabang = df_final.groupBy("nama_cabang") \
    .agg(
        sum("total_penjualan").alias("total_penjualan"),
        count("trx_id").alias("jumlah_transaksi"),
        round(avg("total_penjualan"), 2).alias("rata_rata_transaksi")
    )

df_ringkasan_cabang.show(truncate=False)


df_tidak_laku_jogja = df_final.filter(col("nama_cabang") == "Kopi Nusantara Yogyakarta") \
    .groupBy("nama_menu") \
    .agg(
        sum("qty").alias("total_kuantitas_terjual"),
        sum("total_penjualan").alias("total_pendapatan")
    ) \
    .orderBy(col("total_kuantitas_terjual").asc())

df_tidak_laku_jogja.show(1)

+-------------------------+---------------+----------------+-------------------+
|nama_cabang              |total_penjualan|jumlah_transaksi|rata_rata_transaksi|
+-------------------------+---------------+----------------+-------------------+
|Kopi Nusantara Yogyakarta|84645000       |1305            |64862.07           |
|Kopi Nusantara Magelang  |87138000       |1302            |66926.27           |
|Kopi Nusantara Semarang  |90542000       |1393            |64997.85           |
+-------------------------+---------------+----------------+-------------------+

+---------+-----------------------+----------------+
|nama_menu|total_kuantitas_terjual|total_pendapatan|
+---------+-----------------------+----------------+
|  Menu-13|                    181|         6335000|
+---------+-----------------------+----------------+
only showing top 1 row



Dari ketiga cabang yang terdaftar pada file bernama tugas7_cabang.json , tugas7_transaksi.csv, tugas7_menu.csv, cabang dengan kinerja terbaik merupakan cabang kopi nusantara yang berada di kota semarang, cabang kopi nusantara yang berada di kota semarang dinilai menjadi cabang dengan kinerja terbaik karena hal tersebut dinilai dari total penjualan yang cabang tersebut dapat diraih.

Jika cabang daru kopi nusantara yang berada di semarang merupakan cabang dengan kinerja terbaik, maka cabang kopi nusantara yang berada di kota yogyakarta merupakan cabang dengan kinerja paling buruk. Cabang inni dinilai menjadi cabang yang paling buruk karena total penjualannya lebih rendah  dari cabang lain. Pada cabang ini,menu yang perlu dipromosikan dengan lebih gencar merupakan menu-13. Alasan menu tersebut dinilai perlu dipromosikan lebih gentar karena menu tersebut hanya memiliki kuantitas terjual sebesar 181. Walaupun selisihnya tidak terlalu besar jika dibandingkan dengan menu lain, tetapi tidak dapat dipungkiri kalau menu tersebut memang berada di titik tidak selaris menu yang lain walau perbedaanya hanya sedikit.